# IRC Labels: Governance via Trusted Engine

This notebook demonstrates **label-driven governance** with ClickHouse as the trusted engine.

The architecture:
```
Catalog (classifies)  →  IRC Labels (carries intent)  →  ClickHouse (enforces natively)
  sensitivity=restricted      LoadTableResponse              auto-masking, column blocking
  pii_type=email              labels field                   row policies, audit logging
```

**Key idea**: Labels carry classification intent. The engine reads labels and enforces policies
natively — masking, blocking, audit. One classification, every engine enforces.

**See [demo.ipynb](demo.ipynb)** for non-governance use cases (ownership, discovery, AI agents, FinOps).

In [ ]:
import sys
sys.path.insert(0, 'agent')

from pyiceberg.catalog import load_catalog
import clickhouse_connect

# Connect to catalog (via Labels Proxy)
catalog = load_catalog(
    "unity",
    **{"type": "rest", "uri": "http://labels-proxy:8181", "warehouse": "unity",
       "prefix": "api/2.1/unity-catalog/iceberg"},
)

# Connect to ClickHouse
ch = clickhouse_connect.get_client(host='clickhouse', port=8123)

---
## Part 1: The Problem — Raw Data, No Governance

ClickHouse connects to the catalog via IRC and queries tables directly.
No governance — all data exposed, including PII.

In [ ]:
# Raw query — no governance, all PII exposed
print("=== Raw query: SELECT * FROM healthcare.patients ===")
print("(No governance — all data visible)\n")

result = ch.query("SELECT * FROM healthcare.patients LIMIT 5")
for i, col in enumerate(result.column_names):
    print(f"  {col:15s}", end="")
print()
print("-" * 90)
for row in result.result_rows:
    for val in row:
        print(f"  {str(val):15s}", end="")
    print()

print("\n→ PII fully exposed: names, emails, diagnoses, dates of birth.")
print("  Any user with SELECT access sees everything.")

---
## Part 2: Populate `iceberg_labels` from IRC

Read labels from the catalog via IRC and populate a ClickHouse table.
This simulates what a native `system.iceberg_labels` would do — the engine
parses labels from `LoadTableResponse` and makes them queryable in SQL.

In [ ]:
from governance_engine import populate_iceberg_labels

# Read labels from IRC → populate ClickHouse table
count = populate_iceberg_labels(catalog, ch, "healthcare")
print(f"Loaded {count} labels into iceberg_labels\n")

# Now labels are queryable in SQL
print("=== SELECT * FROM iceberg_labels ===")
result = ch.query("""
    SELECT database, table_name, scope, column_name, label_key, label_value
    FROM iceberg_labels
    ORDER BY table_name, scope DESC, column_name, label_key
""")
print(f"{'database':12s} {'table':18s} {'scope':8s} {'column':12s} {'key':20s} {'value'}")
print("-" * 95)
for row in result.result_rows:
    col = row[3] or '—'
    print(f"{row[0]:12s} {row[1]:18s} {row[2]:8s} {col:12s} {row[4]:20s} {row[5]}")

In [ ]:
# Discovery queries — pure SQL, no Python needed
print("=== Which tables are sensitive? ===")
for row in ch.query("""
    SELECT table_name, label_value as sensitivity
    FROM iceberg_labels
    WHERE label_key = 'sensitivity' AND scope = 'table'
    ORDER BY sensitivity DESC
""").result_rows:
    print(f"  {row[0]:20s} sensitivity={row[1]}")

print("\n=== Which columns contain PII? ===")
for row in ch.query("""
    SELECT table_name, column_name, label_key, label_value
    FROM iceberg_labels
    WHERE label_key IN ('pii_type', 'phi_type')
    ORDER BY table_name, column_name
""").result_rows:
    print(f"  {row[0]:18s} {row[1]:15s} {row[2]}={row[3]}")

print("\n→ All discoverable via SQL. No external tools needed.")

---
## Part 3: Label-Driven Governance — One Function Call

Read labels → generate governed views with masking and column blocking.
ClickHouse enforces natively. No manual policy authoring.

In [ ]:
from governance_engine import apply_label_governance

# One function call: labels → governed views
views = apply_label_governance(catalog, ch, "healthcare", role="analyst")

In [ ]:
# Show the generated DDL — masking driven entirely by labels
for table_name, info in views.items():
    print(f"\n{'='*80}")
    print(f"Table: {table_name} → View: {info['view']}")
    print(f"{'='*80}")
    print(info['ddl'])

---
## Part 4: Analyst vs Admin — Same Table, Different Access

The governed view masks data based on the current ClickHouse user.
Same SQL, different results.

In [ ]:
# Analyst sees governed (masked) data
print("=== Analyst: SELECT * FROM governed_patients ===")
print("(PII masked, restricted columns blocked)\n")

result = ch.query("SELECT * FROM healthcare.governed_patients LIMIT 5")
for i, col in enumerate(result.column_names):
    print(f"  {col:15s}", end="")
print()
print("-" * 90)
for row in result.result_rows:
    for val in row:
        print(f"  {str(val):15s}", end="")
    print()

In [ ]:
# Side-by-side comparison: raw vs governed
print("=== SIDE BY SIDE: Raw vs Governed ===\n")

raw = ch.query("SELECT name, email, diagnosis FROM healthcare.patients LIMIT 4").result_rows
gov = ch.query("SELECT name, email, diagnosis FROM healthcare.governed_patients LIMIT 4").result_rows

print(f"{'RAW':40s} │ {'GOVERNED (analyst role)'}")
print(f"{'name':12s} {'email':18s} {'diagnosis':10s} │ {'name':12s} {'email':18s} {'diagnosis'}")
print("─" * 42 + "┼" + "─" * 45)
for r, g in zip(raw, gov):
    print(f"{str(r[0]):12s} {str(r[1]):18s} {str(r[2]):10s} │ {str(g[0]):12s} {str(g[1]):18s} {str(g[2])}")

print("\n→ Same table. Labels drive the masking. Zero manual configuration.")

In [ ]:
# Safe tables pass through unmodified
print("=== visits_summary: sensitivity=low → no masking ===")
result = ch.query("SELECT * FROM healthcare.governed_visits_summary LIMIT 5")
for col in result.column_names:
    print(f"  {col:20s}", end="")
print()
for row in result.result_rows:
    for val in row:
        print(f"  {str(val):20s}", end="")
    print()
print("\n→ Low sensitivity table — all data visible, no masking applied.")

---
## Part 5: Compliance Inventory — Pure SQL

With labels in ClickHouse, compliance queries are just SQL.

In [ ]:
# HIPAA compliance report
print("=== HIPAA Compliance Inventory ===\n")
hipaa = ch.query("""
    SELECT table_name FROM iceberg_labels 
    WHERE label_key = 'regulatory_scope' AND label_value LIKE '%HIPAA%'
""").result_rows

for (tbl,) in hipaa:
    print(f"  Table: healthcare.{tbl}")
    pii_cols = ch.query(f"""
        SELECT column_name, label_key, label_value FROM iceberg_labels
        WHERE table_name = '{tbl}' AND label_key IN ('pii_type', 'phi_type')
    """).result_rows
    print(f"  PII/PHI columns: {len(pii_cols)}")
    for col_name, lk, lv in pii_cols:
        print(f"    {col_name:15s} {lk}={lv}")
    print()

In [ ]:
# Cross-regulation summary
print("=== Regulatory Coverage ===\n")
for row in ch.query("""
    SELECT label_value as regulation, 
           groupArray(table_name) as tables,
           count() as table_count
    FROM iceberg_labels
    WHERE label_key = 'regulatory_scope'
    GROUP BY regulation
""").result_rows:
    print(f"  {row[0]:10s}  {row[2]} table(s): {row[1]}")

print("\n=== Sensitivity Distribution ===\n")
for row in ch.query("""
    SELECT label_value as sensitivity, count() as col_count
    FROM iceberg_labels
    WHERE label_key = 'sensitivity'
    GROUP BY sensitivity
    ORDER BY col_count DESC
""").result_rows:
    print(f"  {row[0]:15s}  {row[1]} labels")

---
## Part 6: Governance-Aware AI Agent

The AI agent uses labels to refuse unsafe queries and suggest governed alternatives.

In [ ]:
import os
from openai import OpenAI

llm = OpenAI(
    api_key=os.environ.get("LLM_API_KEY", "your-key-here"),
    base_url=os.environ.get("LLM_BASE_URL"),
)
model = os.environ.get("LLM_MODEL", "gpt-4o")

# Build context from labels stored in ClickHouse
label_context = ""
for table_id in catalog.list_tables("healthcare"):
    t = catalog.load_table(table_id)
    label_context += f"\n### Table: healthcare.{table_id[1]}\n"
    label_context += f"  Governed view: healthcare.governed_{table_id[1]}\n"
    for k, v in t.table_labels.items():
        label_context += f"  {k}: {v}\n"
    label_context += "  Columns:\n"
    for field in t.schema().fields:
        cl = t.get_column_label(field.field_id)
        extras = [f'{k}={v}' for k, v in cl.items()]
        extra = f" ({', '.join(extras)})" if extras else ""
        label_context += f"    - {field.name} {field.field_type}{extra}\n"

GOV_SYSTEM = """You are a governance-aware data agent using ClickHouse.
RULES:
1. ALWAYS use governed_* views instead of raw tables.
2. If sensitivity=restricted, WARN the user and explain masking.
3. If columns have pii_type/phi_type, note they will be masked in the governed view.
4. Prefer low-sensitivity tables when possible.
5. Reference specific labels in your reasoning.
6. Generate ClickHouse SQL using governed views."""

def ask_governed(question):
    resp = llm.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": GOV_SYSTEM},
            {"role": "user", "content": f"Tables:\n{label_context}\n---\nQuestion: {question}"},
        ],
        temperature=0, max_tokens=1000,
    )
    return resp.choices[0].message.content

In [ ]:
# Agent routes through governed views
print(ask_governed("How many patients visited cardiology last month?"))

In [ ]:
# Agent handles PII request through governance
print(ask_governed("Show me patient emails for follow-up outreach"))

---
## Architecture Summary

```
┌──────────────┐    IRC + labels    ┌─────────────────────────┐
│   Catalog     │──────────────────►│  ClickHouse             │
│ (Polaris/UC)  │  LoadTableResponse │                         │
│               │  + labels field    │  iceberg_labels table   │
│ label.pii=email                    │  ┌─────────────────────┐│
│ label.sensitivity=restricted       │  │ patients: pii=email ││
└──────────────┘                     │  │ visits: safe        ││
                                     │  └─────────────────────┘│
                                     │           │              │
                                     │  governed_patients view  │
                                     │  ┌─────────────────────┐│
                                     │  │ name → A****        ││
                                     │  │ email → al****@**.com│
                                     │  │ diagnosis → NULL    ││
                                     │  └─────────────────────┘│
                                     └─────────────────────────┘
```

### What this proves

1. **Labels carry governance intent** — catalog classifies, protocol delivers
2. **Engine enforces natively** — masking, blocking, audit in ClickHouse SQL
3. **Zero manual policy authoring** — labels drive everything
4. **Change a label → governance updates** — no DDL changes needed
5. **Any engine can do this** — same pattern works for Trino, Spark, DuckDB

### Native integration path (future)

Today: Python reads labels → populates iceberg_labels → generates views

Tomorrow: ClickHouse natively:
- Parses `labels` from `LoadTableResponse`
- Populates `system.iceberg_labels` system table
- `CREATE GOVERNED VIEW` syntax auto-masks based on labels

The pattern is the same. Labels are the standard. Engine enforces.